# Local File Processing to S3

> **Note:** Notebooks in this directory use CLI commands (`!command`) rather than Python library imports. This keeps workflows simple, reproducible, and avoids environment issues with GDAL/rasterio imports.

Process local GeoTIFF files, convert to Cloud Optimized GeoTIFFs (COGs), and upload to S3.

## Available CLI Tools
- `aws s3 ls / cp` - List and transfer S3 files
- `rio cogeo create` - Convert GeoTIFF to Cloud Optimized GeoTIFF
- `rio cogeo validate` - Validate COG structure
- `rio warp` - Reproject rasters
- `gdalinfo -json` - Inspect GeoTIFF metadata

## Workflow
1. Configure local directory and S3 destination
2. List local .tif files
3. Define filename transformations
4. Preview transformations
5. (Optional) Save mapping to CSV
6. Process and upload files
7. Results summary

---

## Step 1: Configuration

Set your local directory path, S3 destination, and processing options:

In [ ]:
# ========================================
# INPUTS - Edit these for your event
# ========================================

# Local File Path
LOCAL_DIR = '.'  # Change this to your local directory

# S3 Configuration
BUCKET = 'nasa-disasters'    # S3 bucket (DO NOT CHANGE)
DESTINATION_BASE = 'drcs_activations_new'  # Where to save COGs in S3

# Event Details
EVENT_NAME = ''  # Your event name

# Processing Options
OVERWRITE = False          # Set to True to replace existing files in S3
COMPRESSION = 'zstd'       # COG compression profile (zstd, lzw, deflate)
OVERVIEW_LEVEL = '5'       # Number of overview levels
NODATA = '0'               # Nodata value for COG creation

# CRS / Reprojection Options
# ----------------------------
# TARGET_CRS: target CRS for output COG.
#   - 'EPSG:3857' (Web Mercator) — RECOMMENDED for veda-data-airflow / TiTiler
#       Avoids the WGS84 ensemble + lat-first axis bug that breaks
#       rio_stac.get_dataset_geom in airflow's build_stac handler.
#   - 'EPSG:4326' — geographic, but produces ensemble WKT that PROJ-based
#       downstream consumers (rio_stac, build_stac) reject.
#   - None / '' / 'None' — preserve the source CRS (skip rio warp).
# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

# RESAMPLING_METHOD: how rio warp interpolates pixels when reprojecting.
#   - 'near' — nearest neighbor; preserves exact pixel values (use for
#       discrete/categorical data like classification masks or count rasters).
#   - 'bilinear' — smooth 2x2 average; good for continuous physical fields.
#   - 'cubic' — sharper than bilinear for continuous data.
#   - 'average' — area-weighted; preserves means when downsampling.
RESAMPLING_METHOD = 'near'

# CLIP_TO_WEBMERC_EXTENT: clip the output to Web Mercator's valid extent
# (±20037508.34 m, equivalent to ±85.06° latitude).
#   - None (default) — auto-detect via shared_utils.reprojection.needs_webmerc_clip
#       (no-op for regional rasters, automatic for world-extent sources like
#       global Mollweide or polar stereographic).
#   - True / False — force-override the auto-detect decision.
# Has no effect when TARGET_CRS is not EPSG:3857.
CLIP_TO_WEBMERC_EXTENT = None

print(f"Local Directory: {LOCAL_DIR}")
print(f"Event: {EVENT_NAME}")
print(f"Destination: s3://{BUCKET}/{DESTINATION_BASE}/")
print(f"Target CRS:  {TARGET_CRS or 'keep source CRS'}")
print(f"Resampling:  {RESAMPLING_METHOD}")

## Step 2: List Local Files

Scan the local directory for GeoTIFF files:

In [17]:
from pathlib import Path
import os

print("SCANNING LOCAL DIRECTORY")
print("=" * 80)
print(f"\nSearching for .tif files in: {LOCAL_DIR}\n")

local_dir_path = Path(LOCAL_DIR)
if not local_dir_path.exists():
    print(f"ERROR: Directory does not exist: {LOCAL_DIR}")
    print("   Please check your LOCAL_DIR path and try again.")
    local_files = []
else:
    # Find all .tif files recursively (case-insensitive)
    local_files = sorted(
        list(local_dir_path.rglob('*.tif')) + list(local_dir_path.rglob('*.TIF'))
    )

    if local_files:
        print(f"Found {len(local_files)} .tif files\n")
        total_gb = 0
        for i, fp in enumerate(local_files):
            size_gb = fp.stat().st_size / (1024 ** 3)
            total_gb += size_gb
            print(f"  {i+1:3}. {fp.name:<60} ({size_gb:.3f} GB)")
        print(f"\nTotal size: {total_gb:.2f} GB")
    else:
        print("No .tif files found in the specified directory.")
        print("   Check your LOCAL_DIR path.")

print("\n" + "=" * 80)

SCANNING LOCAL DIRECTORY

Searching for .tif files in: .

Found 1 .tif files

    1. low-durability-wood-framed-1.tif                             (0.457 GB)

Total size: 0.46 GB



## Step 3: Define Filename Transformations

Configure how files should be renamed and where each category goes in S3:

In [ ]:
import re
from shared_utils.file_naming import (
    DATETIME_PATTERNS,
    categorize_file as _categorize_file,
    create_output_filename as _create_output_filename,
    extract_datetime_from_filename,
    no_change,
)

# Event-specific category mapping. Pattern -> S3 output subdirectory.
# Edit this for your event; the regex matching logic lives in
# shared_utils.file_naming.categorize_file.
CATEGORIES = {
    r'trueColor|truecolor|true_color|RGB': 'Sentinel-2/trueColor',
    r'colorInfrared|colorIR|color_infrared|CIR': 'Sentinel-2/colorIR',
    r'naturalColor|naturalcolor|natural_color': 'Sentinel-2/naturalColor',
    r'shortwaveIR|SWIR|shortwave': 'Sentinel-2/shortwaveIR',
    r'earlylook': 'AVIRIS',
    r'wood': 'GAIA',
}


def categorize_file(filename):
    """Match filename to a category. Returns the S3 subdirectory."""
    return _categorize_file(filename, CATEGORIES)


def create_output_filename(original_path, event_name):
    """Build a standardized output filename for this event."""
    return _create_output_filename(original_path, event_name, categories=CATEGORIES)


if __name__ == "__main__":
    print("Transformation functions defined")
    print(f"Categories configured: {len(CATEGORIES)}")
    for pattern, directory in CATEGORIES.items():
        print(f"   {directory:<30}  pattern: {pattern}")

## Step 4: Preview Transformations

Review old -> new filename mappings before processing:

In [20]:
# Build processing plan with old -> new mappings
processing_plan = []

if local_files:
    print("PREVIEW TRANSFORMATIONS")
    print("=" * 80)

    uncategorized_count = 0
    for fp in local_files:
        filename = fp.name
        new_name = create_output_filename(str(fp), EVENT_NAME)
        output_dir = categorize_file(filename)
        dest_key = f"{DESTINATION_BASE}/{output_dir}/{new_name}"

        processing_plan.append({
            'local_path': str(fp),
            'filename': filename,
            'new_name': new_name,
            'category': output_dir,
            'dest_key': dest_key,
        })

        tag = f"[{output_dir}]" if output_dir != 'uncategorized' else '[UNCATEGORIZED]'
        print(f"\n  {tag}")
        print(f"  Old: {filename}")
        print(f"  New: s3://{BUCKET}/{dest_key}")

        if output_dir == 'uncategorized':
            uncategorized_count += 1

    print(f"\n{'=' * 80}")
    print(f"Total: {len(processing_plan)} files")
    print(f"  Categorized:   {len(processing_plan) - uncategorized_count}")
    print(f"  Uncategorized: {uncategorized_count}")

    if uncategorized_count:
        print("\nAdd patterns to CATEGORIES dict in Step 3 to categorize unmatched files.")
else:
    print("No files to preview. Check Step 2.")

PREVIEW TRANSFORMATIONS

  [GAIA]
  Old: low-durability-wood-framed-1.tif
  New: s3://nasa-disasters/drcs_activations_new/GAIA/_low-durability-wood-framed-1_day.tif

Total: 1 files
  Categorized:   1
  Uncategorized: 0


## Step 5: Optional CSV Export

Save the filename mapping to a CSV file for your records:

In [21]:
import pandas as pd
from datetime import datetime

SAVE_CSV = True  # Set to False to skip CSV export

if SAVE_CSV and processing_plan:
    df = pd.DataFrame(processing_plan)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path = f'/tmp/{EVENT_NAME}_file_mapping_{timestamp}.csv'
    df.to_csv(csv_path, index=False)
    print(f"Mapping saved to: {csv_path}")
    print(f"Records: {len(df)}")
elif not processing_plan:
    print("No files to export. Check previous steps.")
else:
    print("CSV export disabled (SAVE_CSV = False)")

Mapping saved to: /tmp/_file_mapping_20260518_143656.csv
Records: 1


## Step 6: Process and Upload

For each file: check CRS, reproject if needed, convert to COG, validate, and upload to S3:

In [ ]:
import subprocess, time, json

results = []

# Treat None / "" / "None" as "preserve native CRS" — skip the rio warp step.
_dst_crs = TARGET_CRS if (TARGET_CRS and str(TARGET_CRS).strip().lower() not in ("none", "")) else None

# Web Mercator's valid extent in meters (±85.06° latitude).
_WEBMERC_TE = ('-te', '-20037508.34', '-20037508.34', '20037508.34', '20037508.34')

for i, item in enumerate(processing_plan):
    local_path = item['local_path']
    dest_key = item['dest_key']
    filename = item['filename']
    new_name = item['new_name']

    print(f"[{i+1}/{len(processing_plan)}] Processing: {filename}")
    start = time.time()

    # --- Check if destination already exists ---
    if not OVERWRITE:
        check = subprocess.run(
            ['aws', 's3', 'ls', f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True
        )
        if check.returncode == 0 and check.stdout.strip():
            print(f"  SKIPPED (already exists in S3)\n")
            results.append({'file': filename, 'status': 'skipped', 'time': 0})
            continue

    local_warped = f'/tmp/warped_{new_name}'
    local_cog = f'/tmp/{new_name}'

    try:
        # --- Check CRS with gdalinfo -json ---
        info = subprocess.run(
            ['gdalinfo', '-json', local_path],
            capture_output=True, text=True, check=True
        )
        metadata = json.loads(info.stdout)
        src_wkt = metadata.get('coordinateSystem', {}).get('wkt', '') or ''

        # Determine if reprojection is needed
        if _dst_crs is None:
            needs_reproject = False
        elif not src_wkt.strip():
            print("  WARNING: source has no CRS — skipping reprojection. "
                  "Inspect with `!gdalinfo` and set TARGET_CRS=None or georeference the input.")
            needs_reproject = False
        else:
            needs_reproject = _dst_crs.split(':')[-1] not in src_wkt

        # --- Reproject if needed ---
        if needs_reproject:
            print(f"  Reprojecting to {_dst_crs} (resampling={RESAMPLING_METHOD}, all CPUs)...")
            # gdalwarp gives us -te clipping for world-extent rasters; rio warp does not.
            # -multi: multi-threaded I/O. -wo NUM_THREADS=ALL_CPUS: warp computation
            # uses every core. -co NUM_THREADS=ALL_CPUS: COG creation uses every core.
            warp_cmd = [
                'gdalwarp',
                '-t_srs', _dst_crs,
                '-r', RESAMPLING_METHOD,
                '-multi',
                '-wo', 'NUM_THREADS=ALL_CPUS',
                '-co', 'NUM_THREADS=ALL_CPUS',
                '--config', 'GDAL_NUM_THREADS', 'ALL_CPUS',
                '-overwrite',
            ]
            # Decide whether to clip output extent to Web Mercator's valid domain.
            # None = auto-detect via reprojection.needs_webmerc_clip; True/False = force.
            from shared_utils.reprojection import needs_webmerc_clip
            if CLIP_TO_WEBMERC_EXTENT is None:
                import rasterio
                with rasterio.open(local_path) as _src:
                    _do_clip = needs_webmerc_clip(_src, _dst_crs)
            else:
                _do_clip = bool(CLIP_TO_WEBMERC_EXTENT) and _dst_crs.strip().upper() == 'EPSG:3857'
            if _do_clip:
                warp_cmd += list(_WEBMERC_TE)
            warp_cmd += [local_path, local_warped]
            subprocess.run(warp_cmd, capture_output=True, text=True, check=True)
            cog_input = local_warped
        else:
            cog_input = local_path

        # --- Convert to COG ---
        print(f"  Converting to COG ({COMPRESSION}, all CPUs)...")
        subprocess.run([
            'rio', 'cogeo', 'create', cog_input, local_cog,
            '--cog-profile', COMPRESSION,
            '--overview-level', OVERVIEW_LEVEL,
            '--nodata', NODATA,
            '--overview-resampling', RESAMPLING_METHOD,
            '--co', 'NUM_THREADS=ALL_CPUS',
        ], capture_output=True, text=True, check=True)

        # --- Validate COG ---
        val = subprocess.run(
            ['rio', 'cogeo', 'validate', local_cog],
            capture_output=True, text=True
        )
        is_valid = val.returncode == 0
        if not is_valid:
            print(f"  WARNING: COG validation issue: {val.stdout.strip()}")

        # --- Upload to S3 ---
        print(f"  Uploading to s3://{BUCKET}/{dest_key}...")
        subprocess.run(
            ['aws', 's3', 'cp', local_cog, f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True, check=True
        )

        elapsed = time.time() - start
        status = 'success' if is_valid else 'success (COG validation warning)'
        print(f"  {status} ({elapsed:.1f}s)\n")
        results.append({'file': filename, 'status': status, 'time': elapsed})

    except subprocess.CalledProcessError as e:
        elapsed = time.time() - start
        err_msg = e.stderr[:300] if e.stderr else str(e)
        print(f"  FAILED: {err_msg}\n")
        results.append({'file': filename, 'status': 'failed', 'time': elapsed, 'error': err_msg})

    finally:
        # Cleanup temp files
        for f in [local_warped, local_cog]:
            if os.path.exists(f):
                os.remove(f)

print("=" * 80)
print("Processing complete.")

## Step 7: Results Summary

Display processing statistics:

In [23]:
if results:
    total = len(results)
    success = sum(1 for r in results if 'success' in r['status'])
    failed = sum(1 for r in results if r['status'] == 'failed')
    skipped = sum(1 for r in results if r['status'] == 'skipped')

    print("RESULTS SUMMARY")
    print("=" * 80)
    print(f"\nTotal files:  {total}")
    print(f"  Success:    {success}")
    print(f"  Failed:     {failed}")
    print(f"  Skipped:    {skipped}")

    if total > 0:
        print(f"\nSuccess rate: {(success / total * 100):.1f}%")

    if success > 0:
        times = [r['time'] for r in results if 'success' in r['status']]
        print(f"\nTiming:")
        print(f"  Average: {sum(times) / len(times):.1f}s per file")
        print(f"  Slowest: {max(times):.1f}s")
        print(f"  Total:   {sum(times):.1f}s ({sum(times) / 60:.1f} minutes)")

    if failed > 0:
        print(f"\nFailed files:")
        for r in results:
            if r['status'] == 'failed':
                print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

    print("\n" + "=" * 80)
else:
    print("No results to display. Run Step 6 first.")

RESULTS SUMMARY

Total files:  1
  Success:    1
  Failed:     0
  Skipped:    0

Success rate: 100.0%

Timing:
  Average: 67.5s per file
  Slowest: 67.5s
  Total:   67.5s (1.1 minutes)



## Troubleshooting

- **"Directory does not exist"** - Check `LOCAL_DIR` path is correct; use absolute paths
- **"No .tif files found"** - Verify files have `.tif` extension; check subdirectories
- **Files being skipped** - Set `OVERWRITE = True` in Step 1
- **Processing errors** - Check `aws configure` has valid credentials; verify `/tmp` disk space
- **Wrong CRS** - Inspect with `!gdalinfo /tmp/yourfile.tif` and adjust `TARGET_CRS`
- **COG validation warnings** - Usually harmless; check with `!rio cogeo validate /tmp/yourfile.tif`